# Task 7 — SQL Business Analysis (Gold Layer)

20 business-focused SQL queries over the `sk_dev.gold` datasets (`fact_bookings`, `dim_passenger`, `dim_route`, `dim_airline`, `dim_geography`, `vw_annual_booking_kpis`). Every query is tested and paired with a business interpretation and a suggested management action.

## Q01 — Overall KPI summary

**Business interpretation:** Total net revenue is ≈ $105.9M across 1994–1998 from roughly 7.6k distinct bookings, giving an average booking value of ≈ $13.9k, with passengers active across every region.

**Management action:** Track these four KPIs as the executive baseline and investigate any period that deviates materially from trend.

In [0]:
%sql
SELECT
    ROUND(SUM(net_revenue), 2)                              AS net_revenue,
    COUNT(DISTINCT booking_id)                              AS distinct_bookings,
    COUNT(DISTINCT passenger_id)                            AS active_passengers,
    ROUND(SUM(net_revenue) / COUNT(DISTINCT booking_id), 2) AS average_booking_value
FROM sk_dev.gold.fact_bookings

## Q02 — Monthly revenue trend across all years

**Business interpretation:** Monthly revenue ranges from $1.48M (Dec 1995) to $2.28M (Dec 1998); December is the strongest month of 1994 and 1998, pointing to mild year-end seasonality rather than volatility.

**Management action:** Set monthly revenue targets from this baseline and trigger a commercial review whenever a month breaches the tolerance band.

In [0]:
%sql
SELECT
    YEAR(booking_date)          AS booking_year,
    MONTH(booking_date)         AS booking_month,
    ROUND(SUM(net_revenue), 2)  AS net_revenue,
    COUNT(DISTINCT booking_id)  AS bookings
FROM sk_dev.gold.fact_bookings
GROUP BY 1, 2
ORDER BY 1, 2

## Q03 — Annual revenue with year-over-year change

**Business interpretation:** Revenue dipped −5.8% in 1995, recovered slowly in 1996–1997 (+2.8%, +0.6%) and accelerated +6.1% in 1998 to $22.26M — the best year in the window.

**Management action:** Identify what drove the 1998 acceleration (pricing, capacity, mix) and protect those levers; treat 1995 as the downside scenario in planning.

In [0]:
%sql
WITH annual AS (
    SELECT YEAR(booking_date) AS booking_year,
           SUM(net_revenue)   AS net_revenue
    FROM sk_dev.gold.fact_bookings
    GROUP BY 1
)
SELECT
    booking_year,
    ROUND(net_revenue, 2)                                   AS net_revenue,
    ROUND(LAG(net_revenue) OVER (ORDER BY booking_year), 2) AS prior_year_revenue,
    ROUND((net_revenue - LAG(net_revenue) OVER (ORDER BY booking_year))
        / LAG(net_revenue) OVER (ORDER BY booking_year) * 100, 2) AS yoy_change_pct
FROM annual
ORDER BY booking_year

## Q04 — Revenue by passenger travel purpose

**Business interpretation:** BUSINESS travel leads with $22.86M while CREW is lowest at $20.11M — the spread across purposes is narrow (~12%), so no single purpose dominates the book.

**Management action:** Protect the corporate (BUSINESS) segment with contract fares, and price CREW traffic deliberately since it is likely negotiated/internal volume.

In [0]:
%sql
SELECT
    p.travel_purpose             AS travel_purpose,
    ROUND(SUM(f.net_revenue), 2) AS net_revenue,
    COUNT(DISTINCT f.booking_id) AS bookings
FROM sk_dev.gold.fact_bookings f
JOIN sk_dev.gold.dim_passenger p
  ON f.passenger_id = p.passenger_id
GROUP BY 1
ORDER BY 2 DESC

## Q05 — Top 10 passengers by net revenue

**Business interpretation:** Michael Ali ($302.1k), Patricia Williams ($200.3k) and Charles Nguyen ($192.1k) head the list; the top spenders fly far more than the ~$71k passenger average. The ranking keys on passenger_id because several passengers share the same name — grouping by name alone would merge distinct customers.

**Management action:** Study what drives these repeat high-value fliers (routes, cabins, purpose) and build loyalty/retention offers around that profile.

In [0]:
%sql
SELECT
    p.passenger_id               AS passenger_id,
    p.passenger_name             AS passenger_name,
    p.travel_purpose             AS travel_purpose,
    ROUND(SUM(f.net_revenue), 2) AS net_revenue
FROM sk_dev.gold.fact_bookings f
JOIN sk_dev.gold.dim_passenger p
  ON f.passenger_id = p.passenger_id
GROUP BY 1, 2, 3
ORDER BY 4 DESC
LIMIT 10

## Q06 — Top 10 passengers as a share of total revenue

**Business interpretation:** The top 10 passengers contribute only ≈ 1.9% of total revenue — customer revenue is highly diversified and there is no single-customer dependency.

**Management action:** With concentration this low, prioritize broad acquisition and loyalty programs over individual whale-hunting; still monitor this share as a concentration-risk KPI.

In [0]:
%sql
WITH top10 AS (
    SELECT passenger_id, SUM(net_revenue) AS revenue
    FROM sk_dev.gold.fact_bookings
    GROUP BY 1
    ORDER BY 2 DESC
    LIMIT 10
)
SELECT
    COUNT(*)    AS passengers_in_top10,
    ROUND(SUM(revenue), 2) AS top10_revenue,
    ROUND((SELECT SUM(net_revenue) FROM sk_dev.gold.fact_bookings), 2)
        AS total_revenue,
    ROUND(SUM(revenue) * 100.0
        / (SELECT SUM(net_revenue) FROM sk_dev.gold.fact_bookings), 2)
        AS revenue_share_pct
FROM top10

## Q07 — Top 10 routes by net revenue

**Business interpretation:** The strongest individual route entries peak near $1.29M; one lane name (e.g. JFK-MAD) can span multiple route entries operated with different aircraft/brands, so strong markets can appear more than once in the ranking.

**Management action:** Protect capacity and pricing on the top lanes, and consolidate schedule planning where one market is fragmented across several route entries.

In [0]:
%sql
SELECT
    route_id                     AS route_id,
    route_name                   AS route_name,
    ROUND(SUM(net_revenue), 2)   AS net_revenue,
    COUNT(DISTINCT booking_id)   AS bookings
FROM sk_dev.gold.fact_bookings
GROUP BY 1, 2
ORDER BY 3 DESC
LIMIT 10

## Q08 — Bottom 10 routes by net revenue

**Business interpretation:** The weakest route entries still generate ≈ $0.85M versus ≈ $1.29M at the top — a ~35% gap, so even the bottom of the network contributes substantial revenue and no lane is trivially disposable.

**Management action:** Before cutting capacity, review scheduling and aircraft assignment on bottom lanes and test cabin-mix or fare improvements first.

In [0]:
%sql
SELECT
    route_id                     AS route_id,
    route_name                   AS route_name,
    ROUND(SUM(net_revenue), 2)   AS net_revenue,
    COUNT(DISTINCT booking_id)   AS bookings
FROM sk_dev.gold.fact_bookings
GROUP BY 1, 2
ORDER BY 3 ASC
LIMIT 10

## Q09 — Revenue by route type

**Business interpretation:** ULTRA_LONG_HAUL leads with $29.47M and DOMESTIC is lowest at $23.65M — longer-haul flying generates the majority of revenue despite fewer possible frequencies.

**Management action:** Prioritize wide-body capacity and premium-cabin inventory on long-haul routes while keeping domestic flying positioned as feeder traffic.

In [0]:
%sql
SELECT
    route_type                                        AS route_type,
    ROUND(net_revenue, 2)                             AS net_revenue,
    ROUND(net_revenue * 100.0 / SUM(net_revenue) OVER (), 2) AS revenue_share_pct
FROM (
    SELECT route_type, SUM(net_revenue) AS net_revenue
    FROM sk_dev.gold.fact_bookings
    GROUP BY 1
)
ORDER BY net_revenue DESC

## Q10 — Top 10 airlines by net revenue

**Business interpretation:** AmeriJet leads with $10.78M, but the spread across all ten carriers is tight (~$10.1–10.8M), so no airline dominates the market.

**Management action:** With revenue this evenly clustered, differentiate on service quality and unit cost rather than chasing share alone.

In [0]:
%sql
SELECT
    airline_id                   AS airline_id,
    airline_name                 AS airline_name,
    ROUND(SUM(net_revenue), 2)   AS net_revenue,
    COUNT(DISTINCT booking_id)   AS bookings
FROM sk_dev.gold.fact_bookings
GROUP BY 1, 2
ORDER BY 3 DESC
LIMIT 10

## Q11 — Revenue share of the top 10 airlines

**Business interpretation:** The top 10 airlines are the entire carrier base, so they necessarily hold ~100% of revenue; the meaningful insight is that each airline holds an almost equal ~10% slice — a remarkably even market.

**Management action:** Monitor monthly for the first sign of share shift; in an even market, partnership reach and loyalty integration decide winners, not scale.

In [0]:
%sql
WITH top10 AS (
    SELECT airline_id, SUM(net_revenue) AS revenue
    FROM sk_dev.gold.fact_bookings
    GROUP BY 1
    ORDER BY 2 DESC
    LIMIT 10
)
SELECT
    COUNT(*)    AS airlines_in_top10,
    ROUND(SUM(revenue), 2) AS top10_revenue,
    ROUND((SELECT SUM(net_revenue) FROM sk_dev.gold.fact_bookings), 2)
        AS total_revenue,
    ROUND(SUM(revenue) * 100.0
        / (SELECT SUM(net_revenue) FROM sk_dev.gold.fact_bookings), 2)
        AS revenue_share_pct
FROM top10

## Q12 — Revenue by passenger region

**Business interpretation:** EUROPE leads with $22.95M and AMERICA trails at $19.52M; all five regions sit within ~15% of each other, indicating a well-balanced geographic demand base.

**Management action:** Keep the balanced network, but direct growth investment where penetration is lowest (AMERICA) rather than over-serving the leader.

In [0]:
%sql
SELECT
    passenger_region               AS passenger_region,
    ROUND(SUM(net_revenue), 2)     AS net_revenue,
    COUNT(DISTINCT passenger_id)   AS passengers,
    COUNT(DISTINCT booking_id)     AS bookings
FROM sk_dev.gold.fact_bookings
GROUP BY 1
ORDER BY 2 DESC

## Q13 — Revenue by passenger nation (all nations)

**Business interpretation:** FRANCE is the strongest source market at $5.68M, followed by TURKEY ($4.96M) and NETHERLANDS ($4.74M); no single nation overwhelms the mix.

**Management action:** Replicate the French market playbook (pricing, partnerships, marketing) in mid-tier nations to lift the tail of the country ranking.

In [0]:
%sql
SELECT
    passenger_nation               AS passenger_nation,
    ROUND(SUM(net_revenue), 2)     AS net_revenue,
    COUNT(DISTINCT passenger_id)   AS passengers,
    COUNT(DISTINCT booking_id)     AS bookings
FROM sk_dev.gold.fact_bookings
GROUP BY 1
ORDER BY 2 DESC

## Q14 — Revenue and volume split by refund flag

**Business interpretation:** (refund_flag lives at flight-line level in Silver and is joined on the composite line key.) Lines with no refund flag (N) carry $69.11M across 16,290 lines, while flagged lines (A and R) account for ~35% of lines and ~$36.8M — a material refund exposure.

**Management action:** Quantify refund leakage by flag category and tighten fare rules / change policies where flagged volume is highest.

In [0]:
%sql
SELECT
    s.refund_flag                 AS refund_flag,
    ROUND(SUM(f.net_revenue), 2)  AS net_revenue,
    COUNT(*)                      AS booking_lines,
    COUNT(DISTINCT f.booking_id)  AS bookings
FROM sk_dev.gold.fact_bookings f
JOIN sk_dev.silver.flight_booking s
  ON f.booking_id = s.booking_key
 AND f.line_number = s.line_number
GROUP BY 1
ORDER BY 2 DESC

## Q15 — Average discount by travel purpose

**Business interpretation:** Discounts barely differ by segment — LEISURE gets the deepest average discount (7.03%) but BUSINESS pays nearly the same (6.78%), so purpose-based price discrimination is essentially not happening.

**Management action:** Test segment-specific fare rules: corporate demand is least price-elastic and can sustain smaller discounts, funding deeper leisure promotions.

In [0]:
%sql
SELECT
    p.travel_purpose             AS travel_purpose,
    ROUND(AVG(f.discount), 4)    AS avg_discount,
    COUNT(*)                     AS booking_lines
FROM sk_dev.gold.fact_bookings f
JOIN sk_dev.gold.dim_passenger p
  ON f.passenger_id = p.passenger_id
GROUP BY 1
ORDER BY 2 DESC

## Q16 — Average flight duration (departure → arrival, days) by cabin class

**Business interpretation:** Average duration rounds to 0 days in every cabin class — flights arrive on the same calendar day as departure, and the date-only grain cannot resolve sub-day durations, so cabin classes are indistinguishable on this measure.

**Management action:** If duration analysis matters operationally, capture departure/arrival timestamps (not just dates) in the source pipeline; today this metric can only separate multi-day itineraries.

In [0]:
%sql
SELECT
    cabin_class                                          AS cabin_class,
    ROUND(AVG(DATEDIFF(arrival_date, departure_date)), 2) AS avg_duration_days,
    MIN(DATEDIFF(arrival_date, departure_date))          AS min_days,
    MAX(DATEDIFF(arrival_date, departure_date))          AS max_days
FROM sk_dev.gold.fact_bookings
GROUP BY 1
ORDER BY 2 DESC

## Q17 — Booking count and revenue by booking priority

**Business interpretation:** Volume and revenue are nearly flat across priorities; 2-HIGH leads (1,604 bookings, $22.22M) while 1-URGENT is not the largest block — priority labels do not currently translate into outsized value.

**Management action:** Audit whether 1-URGENT handling (SLAs, fees) is worth its cost, since urgent volume is not proportionally more valuable; consider monetizing priority properly.

In [0]:
%sql
SELECT
    booking_priority              AS booking_priority,
    COUNT(DISTINCT booking_id)    AS bookings,
    ROUND(SUM(net_revenue), 2)    AS net_revenue
FROM sk_dev.gold.fact_bookings
GROUP BY 1
ORDER BY 1

## Q18 — Monthly breakdown of booking status

**Business interpretation:** Status F (fulfilled/finalized) dominates overall (~75% of bookings, $79.21M), with open (O) and cancelled (C) bookings (~10% of bookings) visible month by month, so any deterioration in cancellation or open backlog shows up immediately in this view.

**Management action:** Track the monthly C (cancelled) share as an early-warning KPI and investigate pricing or schedule-reliability issues in months where it rises.

In [0]:
%sql
SELECT
    YEAR(booking_date)          AS booking_year,
    MONTH(booking_date)         AS booking_month,
    booking_status              AS booking_status,
    COUNT(DISTINCT booking_id)  AS bookings,
    ROUND(SUM(net_revenue), 2)  AS net_revenue
FROM sk_dev.gold.fact_bookings
GROUP BY 1, 2, 3
ORDER BY 1, 2, 3

## Q19 — Route revenue ranking within each travel purpose

**Business interpretation:** Each segment values different lanes — LEISURE tops out on PEK-MNL ($330.9k), STUDENT on MEX-DEL ($325.9k), MILITARY on DXB-DEL ($324.4k), CREW on HKG-ACC ($309.1k) and BUSINESS on SIN-MAD ($319.2k) — so route appeal is genuinely segment-specific, with LOS-MIA appearing in three segments.

**Management action:** Align marketing and capacity by segment: promote each segment's leading lanes to that segment and schedule aircraft accordingly.

In [0]:
%sql
WITH route_purpose AS (
    SELECT
        p.travel_purpose             AS travel_purpose,
        f.route_id                   AS route_id,
        f.route_name                 AS route_name,
        SUM(f.net_revenue)           AS net_revenue
    FROM sk_dev.gold.fact_bookings f
    JOIN sk_dev.gold.dim_passenger p
      ON f.passenger_id = p.passenger_id
    GROUP BY 1, 2, 3
),
ranked AS (
    SELECT *,
        DENSE_RANK() OVER (
            PARTITION BY travel_purpose ORDER BY net_revenue DESC
        ) AS revenue_rank
    FROM route_purpose
)
SELECT
    travel_purpose,
    revenue_rank,
    route_id,
    route_name,
    ROUND(net_revenue, 2) AS net_revenue
FROM ranked
WHERE revenue_rank <= 3
ORDER BY travel_purpose, revenue_rank

## Q20 — Passenger geography vs airline geography by region

**Business interpretation:** Passenger-side revenue peaks in EUROPE ($22.95M), but AMERICA is the only region whose carriers capture more than their home demand generates (airline-side $21.22M vs passenger-side $19.52M) — demand and carrier domicile genuinely diverge by region.

**Management action:** Use these mismatches to guide alliance, codeshare, or market-entry decisions where demand is strong but local carriers capture relatively less of the spend.

In [0]:
%sql
WITH passenger_geo AS (
    SELECT passenger_region AS region, SUM(net_revenue) AS passenger_revenue
    FROM sk_dev.gold.fact_bookings
    GROUP BY 1
),
airline_geo AS (
    SELECT airline_region AS region, SUM(net_revenue) AS airline_revenue
    FROM sk_dev.gold.fact_bookings
    GROUP BY 1
)
SELECT
    COALESCE(p.region, a.region)                    AS region,
    ROUND(COALESCE(p.passenger_revenue, 0), 2)      AS passenger_region_revenue,
    ROUND(COALESCE(a.airline_revenue, 0), 2)        AS airline_region_revenue
FROM passenger_geo p
FULL OUTER JOIN airline_geo a
  ON p.region = a.region
ORDER BY passenger_region_revenue DESC